# 🏥 Healthcare Veri Seti — MVP v2

**Veri Seti:** [Kaggle — Healthcare Dataset (prasad22)](https://www.kaggle.com/datasets/prasad22/healthcare-dataset)  
**Hedef:** Hasta test sonucunu tahmin etmek (Normal / Abnormal / Inconclusive)

## Proje Akışı
1. Kütüphane ve Veri Yükleme  
2. Keşifsel Veri Analizi (EDA)  
3. Veri Ön İşleme  
4. Model Eğitimi ve Test Skoru  
5. Model Kaydetme  
6. Streamlit Uygulaması (UI + Deploy)

## 1. Kütüphane ve Veri Yükleme

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report
import joblib
import os

print('Kütüphaneler yüklendi.')

Kütüphaneler yüklendi.


In [3]:
df = pd.read_csv('../data/healthcare_dataset.csv')

print(f'Satır sayısı : {df.shape[0]:,}')
print(f'Sütun sayısı : {df.shape[1]}')
df.head()

Satır sayısı : 55,500
Sütun sayısı : 15


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


## 2. Keşifsel Veri Analizi (EDA)

In [4]:
print('=== Değişken Tipleri & Eksik Değerler ===')
info = pd.DataFrame({
    'dtype'       : df.dtypes,
    'null_sayisi' : df.isnull().sum(),
    'null_yuzde'  : (df.isnull().sum() / len(df) * 100).round(2)
})
print(info)

=== Değişken Tipleri & Eksik Değerler ===
                      dtype  null_sayisi  null_yuzde
Name                 object            0         0.0
Age                   int64            0         0.0
Gender               object            0         0.0
Blood Type           object            0         0.0
Medical Condition    object            0         0.0
Date of Admission    object            0         0.0
Doctor               object            0         0.0
Hospital             object            0         0.0
Insurance Provider   object            0         0.0
Billing Amount      float64            0         0.0
Room Number           int64            0         0.0
Admission Type       object            0         0.0
Discharge Date       object            0         0.0
Medication           object            0         0.0
Test Results         object            0         0.0


In [5]:
print('=== Sayısal Değişkenler ===')
print(df.describe())

=== Sayısal Değişkenler ===
                Age  Billing Amount   Room Number
count  55500.000000    55500.000000  55500.000000
mean      51.539459    25539.316097    301.134829
std       19.602454    14211.454431    115.243069
min       13.000000    -2008.492140    101.000000
25%       35.000000    13241.224652    202.000000
50%       52.000000    25538.069376    302.000000
75%       68.000000    37820.508436    401.000000
max       89.000000    52764.276736    500.000000


In [6]:
print('=== Kategorik Değişkenler ===')
cat_cols = ['Gender', 'Blood Type', 'Medical Condition',
            'Insurance Provider', 'Admission Type', 'Medication', 'Test Results']

for col in cat_cols:
    print(f'\n{col}:')
    print(df[col].value_counts())

=== Kategorik Değişkenler ===

Gender:
Gender
Male      27774
Female    27726
Name: count, dtype: int64

Blood Type:
Blood Type
A-     6969
A+     6956
AB+    6947
AB-    6945
B+     6945
B-     6944
O+     6917
O-     6877
Name: count, dtype: int64

Medical Condition:
Medical Condition
Arthritis       9308
Diabetes        9304
Hypertension    9245
Obesity         9231
Cancer          9227
Asthma          9185
Name: count, dtype: int64

Insurance Provider:
Insurance Provider
Cigna               11249
Medicare            11154
UnitedHealthcare    11125
Blue Cross          11059
Aetna               10913
Name: count, dtype: int64

Admission Type:
Admission Type
Elective     18655
Urgent       18576
Emergency    18269
Name: count, dtype: int64

Medication:
Medication
Lipitor        11140
Ibuprofen      11127
Aspirin        11094
Paracetamol    11071
Penicillin     11068
Name: count, dtype: int64

Test Results:
Test Results
Abnormal        18627
Normal          18517
Inconclusive    18356


## 3. Veri Ön İşleme

In [7]:
df_model = df.copy()

# 1. TARİH FEATURE: Sadece 'yatis_gun' alıyoruz
# Fatura miktarını tahmin etmek için yatış süresi en kritik bilgidir.
df_model['Date of Admission'] = pd.to_datetime(df_model['Date of Admission'])
df_model['Discharge Date']    = pd.to_datetime(df_model['Discharge Date'])
df_model['yatis_gun']         = (df_model['Discharge Date'] - df_model['Date of Admission']).dt.days

# 2. LABEL ENCODING: Sadece 'Medical Condition' için
le = LabelEncoder()
df_model['Medical Condition_enc'] = le.fit_transform(df_model['Medical Condition'])

# 3. HEDEF VE FEATURE BELİRLEME
# Hedef (y): Fatura Tutarı
# Özellikler (X): Tıbbi Durum ve Yatış Süresi
features = ['Medical Condition_enc', 'yatis_gun']

X = df_model[features]
y = df_model['Billing Amount']

print(f'Yeni Feature Listesi: {features}')
print(f'Feature sayısı : {X.shape[1]}')
print(f'Örnek sayısı   : {X.shape[0]:,}')

Yeni Feature Listesi: ['Medical Condition_enc', 'yatis_gun']
Feature sayısı : 2
Örnek sayısı   : 55,500


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Eğitim seti: {X_train.shape[0]:,} satır')
print(f'Test seti  : {X_test.shape[0]:,} satır')

Eğitim seti: 44,400 satır
Test seti  : 11,100 satır


## 4. Model Eğitimi ve Test Skoru

**Random Forest Classifier** — birden fazla karar ağacının oylamasıyla tahmin üretir. Yüksek doğruluk, düşük overfitting.

In [9]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_sc, y_train)

print(f"Model Test Skoru (R2): {model.score(X_test_sc, y_test):.4f}")

Model Test Skoru (R2): -0.0031


## 5. Model Kaydetme

In [10]:
# notebooks klasörü yoksa oluştur
os.makedirs("notebooks", exist_ok=True)

# scaler'ı kaydet
joblib.dump(scaler, "notebooks/scaler.pkl")

print("Scaler başarıyla kaydedildi.")

Scaler başarıyla kaydedildi.


In [11]:
joblib.dump(model,  'notebooks/model_mvp.pkl')
joblib.dump(scaler, 'notebooks/scaler.pkl')      # İstediğin scaler dosyası
joblib.dump(le,     'notebooks/le_medical.pkl') # Kategorik dönüşüm için

print("Tüm dosyalar 'notebooks/' klasörüne başarıyla kaydedildi.")

Tüm dosyalar 'notebooks/' klasörüne başarıyla kaydedildi.
